In [0]:
%sql
CREATE VIEW 03_gold.gold.kpi_total_revenue_using_data_cube AS
SELECT ROUND(SUM(line_revenue_usd), 2) AS kpi_total_revenue_usd
FROM 03_gold.gold.customer_data_cube;

In [0]:
%sql
CREATE VIEW 03_gold.gold.kpi_total_quantity_using_data_cube AS
SELECT SUM(quantity) AS kpi_total_quantity_sold
FROM 03_gold.gold.customer_data_cube;

In [0]:
%sql
CREATE VIEW 03_gold.gold.kpi_avg_invoice_value_using_data_cube AS
SELECT ROUND(AVG(invoice_total_usd), 2) AS kpi_avg_invoice_value_usd
FROM (
    SELECT invoice_id, SUM(line_revenue_usd) AS invoice_total_usd
    FROM 03_gold.gold.customer_data_cube
    GROUP BY invoice_id
) t;

In [0]:
%sql
CREATE VIEW 03_gold.gold.kpi_top_customers_using_data_cube AS
SELECT
    cd.customer_id,
    c.customer_name,
    ROUND(SUM(cd.line_revenue_usd), 2) AS total_revenue_usd
FROM 03_gold.gold.customer_data_cube cd
LEFT JOIN 03_gold.gold.dim_customer c ON cd.customer_id = c.customer_id
GROUP BY cd.customer_id, c.customer_name;

In [0]:
%sql
CREATE VIEW 03_gold.gold.kpi_top_products_using_data_cube AS
SELECT
    cd.product_id,
    p.product_name,
    p.category,
    ROUND(SUM(cd.line_revenue_usd), 2) AS total_revenue_usd
FROM 03_gold.gold.customer_data_cube cd
LEFT JOIN 03_gold.gold.dim_products p ON cd.product_id = p.product_id
GROUP BY cd.product_id, p.product_name, p.category;

In [0]:
%sql
CREATE VIEW 03_gold.gold.kpi_cancellation_rate_using_data_cube AS
SELECT
    COUNT(DISTINCT invoice_id) AS total_invoices,
    COUNT(DISTINCT CASE WHEN invoice_status = 'Cancelled' THEN invoice_id END) AS cancelled_invoices,
    ROUND(
        COUNT(DISTINCT CASE WHEN invoice_status = 'Cancelled' THEN invoice_id END) * 100.0 / COUNT(DISTINCT invoice_id), 2
    ) AS cancellation_rate_pct
FROM 03_gold.gold.customer_data_cube;

In [0]:
%sql
CREATE VIEW 03_gold.gold.kpi_discount_impact_using_data_cube AS
SELECT
    ROUND(SUM((discount * rate_to_usd) * quantity), 2)  AS kpi_total_discount_given_usd,
    ROUND(AVG(discount * rate_to_usd), 2)               AS kpi_avg_discount_per_line_usd
FROM 03_gold.gold.customer_data_cube;

In [0]:
%sql
CREATE VIEW 03_gold.gold.kpi_revenue_by_region_using_data_cube AS
SELECT
    region,
    ROUND(SUM(line_revenue_usd), 2) AS total_revenue_usd
FROM 03_gold.gold.customer_data_cube
GROUP BY region;

In [0]:
%sql
CREATE VIEW 03_gold.gold.kpi_invoices_per_customer_using_data_cube AS
SELECT
    cd.customer_id,
    c.customer_name,
    COUNT(DISTINCT cd.invoice_id) AS invoice_count
FROM 03_gold.gold.customer_data_cube cd
LEFT JOIN 03_gold.gold.dim_customer c ON cd.customer_id = c.customer_id
GROUP BY cd.customer_id, c.customer_name;